In [1]:
import pandas as pd
import geopandas as gpd
import unicodedata
import re
from pathlib import Path

In [4]:
# Dataset preparation
# - remove thousand dots
# - replace decimal commas with dots
# - convert numeric-like strings to numbers; leave strings with letters/% as text
# - normalize province names in-place (overwrite 'prov')
# - hardcoded some provinces names for future merging 
# - save cleaned CSV


INPUT_FILE = "../data/ECEC_labour_2022 dataset.xlsx"  
SHEET = "Sheet1"
OUTDIR = Path("../data")

# Matches dots that are thousand separators: a dot between digits and followed by exactly 3 digits
_THOUSAND_DOT = re.compile(r"(?<=\d)\.(?=\d{3}(\D|$))")

def clean_number_string(x):
    
    if pd.isna(x):
        return x
    s = str(x).strip()
    if s == "":
        return s
    if re.search(r"[A-Za-z%]", s):
        return s.replace("\u00A0", "").strip()
    s = _THOUSAND_DOT.sub("", s)
    s = s.replace("\u00A0", "").replace(" ", "")
    s = s.replace(",", ".")
    return s

def convert_object_numeric(df: pd.DataFrame, threshold: float = 0.7) -> pd.DataFrame:
    
    df = df.copy()
    num_pattern = re.compile(r"^-?\d+(\.\d+)?$")
    for c in df.columns:
        if df[c].dtype == "object":
            cleaned = df[c].map(clean_number_string)
            non_na = cleaned[~pd.isna(cleaned)]
            is_numlike = non_na.map(lambda v: bool(num_pattern.fullmatch(str(v))))
            ratio = (is_numlike.mean() if len(non_na) else 0.0)
            if ratio >= threshold:
                df[c] = pd.to_numeric(cleaned, errors="coerce")
            else:
                df[c] = cleaned
    return df

def norm(s):
    s = str(s).strip()
    s = unicodedata.normalize('NFKD', s).encode('ascii','ignore').decode('ascii')
    s = s.replace('-', ' ').replace('/', ' ').replace("'", '').replace('’','')
    s = ' '.join(s.split()).lower()
    return s


df = pd.read_excel(INPUT_FILE, sheet_name=SHEET)
df_clean = convert_object_numeric(df)

if 'prov' in df_clean.columns:
    df_clean['prov'] = df_clean['prov'].apply(norm)
    map = {
        'valle daosta vallee daoste': 'aosta',
        'reggio calabria': 'reggio di calabria',
        'bolzano bozen': 'bolzano',
    }
    df_clean['prov'] = df_clean['prov'].replace(map)

csv_path = OUTDIR / "cleaned_ECEC_labour_2022 dataset.csv"
df_clean.to_csv(csv_path, index=False, encoding="utf-8")

In [ ]:
# Merging csv and geojson into a single file (still geojson to preserve geometry of provinces)

def norm(s):
    s = str(s).strip()
    s = unicodedata.normalize('NFKD', s).encode('ascii','ignore').decode('ascii')
    s = s.replace('-', ' ').replace('/', ' ').replace("'", '').replace('’','')
    s = ' '.join(s.split()).lower()
    return s

csv_path = r"..\data\cleaned_ECEC_labour_2022 dataset.csv"
df = pd.read_csv(csv_path)
df['join_key'] = df['prov'].apply(norm)


gjson_path = r"..\data\georef-italy-provincia.geojson"
gdf = gpd.read_file(gjson_path)

gdf['join_key'] = gdf['prov_name'].apply(norm)

manual_map = {
    'verbano cusio ossola': 'verbano cusio ossola',
    'valle daosta vallee daoste': 'valle daosta vallee daoste',
}
gdf['join_key'] = gdf['join_key'].replace(manual_map)

merged = gdf.merge(df, on='join_key', how='left')

unmatched = merged[merged['population'].isna()][['prov_name']]
print('Unmatched provinces:', unmatched.shape[0])
if unmatched.shape[0] > 0 :
    print(unmatched.head())

merged.drop(['join_key'],axis=1,inplace=True)

out_path = r"..\data\merged_ecec_province.geojson"
merged.to_file(out_path, driver='GeoJSON')

Unmatched provinces: 0
